# QA BT - Tagalog (tl) for contraTICO NER Extension

Answers NER entity-aware questions on Tagalog perturbed text.
Processes all 8 perturbation types × 84 rows = 672 QA calls.

## Environment Setup

In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
elif IN_KAGGLE:
    print('Running on Kaggle')
else:
    print('Running locally')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'torch', 'accelerate'], check=True)
print('Dependencies installed!')

In [ ]:
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()

print(f'Project root: {PROJECT_ROOT}')

## Path Configuration

In [ ]:
RESULTS_DIR = f"{PROJECT_ROOT}/results Qwen3B baseline"
EXTENSION_DIR = f"{RESULTS_DIR}/contratico/ner-extension"
CONTRATICO_CODE_DIR = f"{EXTENSION_DIR}/code"
CONTRATICO_DATA_DIR = f"{PROJECT_ROOT}/contratico/en-tl"

QG_OUTPUT = f"{EXTENSION_DIR}/QG/qg_entity_aware.jsonl"
QA_SOURCE_OUTPUT = f"{EXTENSION_DIR}/QA/source.jsonl"
QA_BT_OUTPUT_DIR = f"{EXTENSION_DIR}/QA/bt/tl"

LANG = "tl"
SAMPLE_SIZE = 125
SEED = 42
PERTURBATIONS = [
    "alteration", "expansion_impact", "expansion_noimpact",
    "intensifier", "omission", "spelling", "synonym", "word_order",
]

os.makedirs(QA_BT_OUTPUT_DIR, exist_ok=True)

print(f"Language: {LANG}")
print(f"QG input: {QG_OUTPUT} (exists: {os.path.exists(QG_OUTPUT)})")
print(f"QA source: {QA_SOURCE_OUTPUT} (exists: {os.path.exists(QA_SOURCE_OUTPUT)})")
print(f"contraTICO data: {CONTRATICO_DATA_DIR}")
print(f"Output: {QA_BT_OUTPUT_DIR}")

## QA BT - Tagalog (tl)

Process all 8 perturbation types.

In [ ]:
for pert in PERTURBATIONS:
    contratico_file = f"{CONTRATICO_DATA_DIR}/{pert}.jsonl"
    output_file = f"{QA_BT_OUTPUT_DIR}/{pert}.jsonl"

    if not os.path.exists(contratico_file):
        print(f"✗ SKIP {pert}: file not found")
        continue

    print(f"\n{'='*50}")
    print(f"Processing: {pert}")
    print(f"{'='*50}")

    cmd = [
        sys.executable, "-u",
        f"{CONTRATICO_CODE_DIR}/qa_entity_contratico.py",
        "--qg_path", QG_OUTPUT,
        "--qa_source_path", QA_SOURCE_OUTPUT,
        "--contratico_path", contratico_file,
        "--lang", LANG,
        "--output_path", output_file,
        "--sample_size", str(SAMPLE_SIZE),
        "--seed", str(SEED),
    ]

    subprocess.run(cmd, check=True)
    print(f"✓ {pert} complete!")

print(f"\n{'='*50}")
print("All perturbations for tl complete!")
print(f"{'='*50}")

## Verification

In [ ]:
import json

total = 0
for pert in PERTURBATIONS:
    path = f"{QA_BT_OUTPUT_DIR}/{pert}.jsonl"
    if os.path.exists(path):
        with open(path) as f:
            n = sum(1 for _ in f)
        print(f"✓ {pert}: {n} rows")
        total += n
    else:
        print(f"✗ {pert}: NOT FOUND")

print(f"\nTotal tl rows: {total}")